In [1]:
# 1) Import libraries

import os
import numpy as np
import pandas as pd
import joblib
from IPython.display import display

In [2]:
# 2) Load saved model and preprocessing artifacts

model_path = "../models/best_nids_model.joblib"
artifacts_path = "../data/processed/preprocessing_artifacts.joblib"

best_model = joblib.load(model_path)
artifacts = joblib.load(artifacts_path)

print("Model and preprocessing artifacts loaded successfully.")

Model and preprocessing artifacts loaded successfully.


In [3]:
# 3) Extract preprocessing objects

scaler = artifacts["scaler"]
label_encoder = artifacts["label_encoder"]
selected_columns = artifacts["selected_columns"]
corr_drop = artifacts["corr_drop"]
zero_var_cols = artifacts["zero_var_cols"]
train_means = artifacts["train_means"]

print("Artifacts extracted successfully.")
print(f"Number of selected columns: {len(selected_columns)}")

Artifacts extracted successfully.
Number of selected columns: 47


In [4]:
# 4) Load new data

new_data_path = "../data/raw/Week_filtered.csv"

new_data = pd.read_csv(new_data_path)
raw_data_for_signature = new_data.copy()

print("New data shape:", new_data.shape)
display(new_data.head())

New data shape: (543734, 79)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,115550790,31,33,318,38380,318,0,10.258065,57.114486,...,32,29534.636364,20922.826407,92617,22941,1.002161e+07,8209.967001,10025078,9998039,BENIGN
1,443,5292756,6,4,611,168,517,0,101.833333,204.305082,...,20,0.000000,0.000000,0,0,0.000000e+00,0.000000,0,0,BENIGN
2,21,3,2,0,14,0,14,0,7.000000,9.899495,...,32,0.000000,0.000000,0,0,0.000000e+00,0.000000,0,0,FTP-Patator
3,54045,41,1,1,2,6,2,2,2.000000,0.000000,...,24,0.000000,0.000000,0,0,0.000000e+00,0.000000,0,0,PortScan
4,80,1023,2,0,0,0,0,0,0.000000,0.000000,...,32,0.000000,0.000000,0,0,0.000000e+00,0.000000,0,0,DoS Hulk


In [5]:
# 5) Clean column names

new_data.columns = new_data.columns.str.strip()
raw_data_for_signature.columns = raw_data_for_signature.columns.str.strip()

print("Column names cleaned.")

Column names cleaned.


In [6]:
# 6) Signature-based detection rules for hybrid NIDS

def signature_detection(row):
    alerts = []

    def get_value(col, default=0):
        if col in row.index:
            value = row[col]
            if pd.isna(value):
                return default
            return value
        return default

    # Rule 1: Possible DoS / DDoS on web ports
    if (
        get_value("Destination Port", 0) in [80, 443]
        and get_value("Flow Packets/s", 0) > 10000
    ):
        alerts.append("Possible DoS/DDoS pattern detected")

    # Rule 2: SYN flood
    if get_value("SYN Flag Count", 0) > 20:
        alerts.append("Possible SYN flood detected")

    # Rule 3: Port scan behavior
    if (
        get_value("Flow Duration", 0) < 50000
        and get_value("Total Fwd Packets", 0) < 5
        and get_value("Total Backward Packets", 0) < 5
    ):
        alerts.append("Possible port scanning activity")

    # Rule 4: Suspiciously high traffic rate
    if get_value("Flow Bytes/s", 0) > 1000000:
        alerts.append("Suspiciously high byte rate")

    # Rule 5: Possible FTP brute-force behavior
    if (
        get_value("Destination Port", 0) == 21
        and get_value("Flow Packets/s", 0) > 500
    ):
        alerts.append("Possible FTP brute-force activity")

    if alerts:
        return "INTRUSION", "; ".join(alerts)

    return "BENIGN", "No signature rule matched"

In [7]:
# 7) Drop label column if it exists

possible_label_cols = ["Label", "label", "Class", "Category", "Attack"]

existing_label_cols = [col for col in possible_label_cols if col in new_data.columns]

if existing_label_cols:
    new_data = new_data.drop(columns=existing_label_cols)
    print("Dropped label columns:", existing_label_cols)
else:
    print("No label column found.")

Dropped label columns: ['Label']


In [8]:
# 8) One-hot encode categorical columns if any

categorical_cols = new_data.select_dtypes(include=["object", "category"]).columns.tolist()

if categorical_cols:
    print("Categorical columns found:", categorical_cols)
    new_data = pd.get_dummies(new_data, columns=categorical_cols, drop_first=False)
    print("One-hot encoding applied.")
else:
    print("No categorical columns found.")

No categorical columns found.


In [9]:
# 9) Remove zero variance columns if they exist

existing_zero_var_cols = [col for col in zero_var_cols if col in new_data.columns]

if existing_zero_var_cols:
    new_data = new_data.drop(columns=existing_zero_var_cols)
    print("Dropped zero-variance columns:", len(existing_zero_var_cols))
else:
    print("No zero-variance columns to drop.")

No zero-variance columns to drop.


In [10]:
# 10) Remove correlation-dropped columns if present

existing_corr_drop = [col for col in corr_drop if col in new_data.columns]

if existing_corr_drop:
    new_data = new_data.drop(columns=existing_corr_drop)
    print("Dropped correlation-based columns:", len(existing_corr_drop))
else:
    print("No correlation-dropped columns to remove.")

Dropped correlation-based columns: 23


In [11]:
# 11) Align columns with training features

for col in selected_columns:
    if col not in new_data.columns:
        new_data[col] = 0

extra_cols = [col for col in new_data.columns if col not in selected_columns]
if extra_cols:
    new_data = new_data.drop(columns=extra_cols)
    print("Dropped extra columns:", len(extra_cols))

new_data = new_data[selected_columns]

print("Column alignment complete.")
print("Aligned shape:", new_data.shape)

Dropped extra columns: 8
Column alignment complete.
Aligned shape: (543734, 47)


In [12]:
# 12) Handle inf / missing values using train means

new_data = new_data.replace([np.inf, -np.inf], np.nan)

for col in new_data.columns:
    if col in train_means.index:
        new_data[col] = new_data[col].fillna(train_means[col])
    else:
        new_data[col] = new_data[col].fillna(0)

print("Missing and infinite values handled.")

Missing and infinite values handled.


In [13]:
# 13) Hybrid prediction: Signature first, ML second

hybrid_results = []

for i in range(len(new_data)):
    raw_row = raw_data_for_signature.iloc[i]
    processed_row = new_data.iloc[[i]]  # DataFrame with one row

    # Step 1: Signature-based detection
    sig_status, sig_reason = signature_detection(raw_row)

    # Step 2: If signature matches, mark as attack
    if sig_status == "INTRUSION":
        final_label = "Signature_Detected_Attack"
        traffic_status = "INTRUSION"
        detection_method = "Signature-Based"

    else:
        # Step 3: Otherwise apply anomaly-based ML model
        X_one = scaler.transform(processed_row)
        X_one = X_one.astype(np.float32)

        pred_encoded = best_model.predict(X_one)[0]
        pred_label = label_encoder.inverse_transform([pred_encoded])[0]

        final_label = pred_label
        traffic_status = "BENIGN" if pred_label == "BENIGN" else "INTRUSION"
        detection_method = "Anomaly-Based"
        sig_reason = "No signature match; predicted by ML model"

    hybrid_results.append({
        "Detection_Method": detection_method,
        "Predicted_Intrusion_Type": final_label,
        "Traffic_Status": traffic_status,
        "Reason": sig_reason
    })

print("Hybrid prediction complete.")

Hybrid prediction complete.


In [14]:
# 14) Create output table

results_df = raw_data_for_signature.copy().reset_index(drop=True)
hybrid_df = pd.DataFrame(hybrid_results)

results_df = pd.concat([results_df, hybrid_df], axis=1)

print("Results preview:")
display(results_df.head(20))

Results preview:


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Detection_Method,Predicted_Intrusion_Type,Traffic_Status,Reason
0,80,115550790,31,33,318,38380,318,0,10.258065,57.114486,...,22941,1.002161e+07,8209.967001,10025078,9998039,BENIGN,Anomaly-Based,BENIGN,BENIGN,No signature match; predicted by ML model
1,443,5292756,6,4,611,168,517,0,101.833333,204.305082,...,0,0.000000e+00,0.000000,0,0,BENIGN,Anomaly-Based,BENIGN,BENIGN,No signature match; predicted by ML model
2,21,3,2,0,14,0,14,0,7.000000,9.899495,...,0,0.000000e+00,0.000000,0,0,FTP-Patator,Signature-Based,Signature_Detected_Attack,INTRUSION,Possible port scanning activity; Suspiciously ...
3,54045,41,1,1,2,6,2,2,2.000000,0.000000,...,0,0.000000e+00,0.000000,0,0,PortScan,Signature-Based,Signature_Detected_Attack,INTRUSION,Possible port scanning activity
4,80,1023,2,0,0,0,0,0,0.000000,0.000000,...,0,0.000000e+00,0.000000,0,0,DoS Hulk,Signature-Based,Signature_Detected_Attack,INTRUSION,Possible port scanning activity
5,53,48119,2,2,60,92,30,30,30.000000,0.000000,...,0,0.000000e+00,0.000000,0,0,BENIGN,Signature-Based,Signature_Detected_Attack,INTRUSION,Possible port scanning activity
6,63286,60,1,1,6,6,6,6,6.000000,0.000000,...,0,0.000000e+00,0.000000,0,0,BENIGN,Signature-Based,Signature_Detected_Attack,INTRUSION,Possible port scanning activity
7,80,71548585,8,5,56,11601,20,0,7.000000,5.656854,...,1040891,7.030000e+07,0.000000,70300000,70300000,DDoS,Anomaly-Based,DDoS,INTRUSION,No signature match; predicted by ML model
8,443,588453,15,10,974,7387,469,0,64.933333,134.265976,...,0,0.000000e+00,0.000000,0,0,BENIGN,Anomaly-Based,BENIGN,BENIGN,No signature match; predicted by ML model
9,80,57772,3,6,26,11607,20,0,8.666667,10.263203,...,0,0.000000e+00,0.000000,0,0,DDoS,Anomaly-Based,DDoS,INTRUSION,No signature match; predicted by ML model


In [15]:
# 15) Summary counts

print("Traffic Status Summary:")
print(results_df["Traffic_Status"].value_counts())

print("\nDetection Method Summary:")
print(results_df["Detection_Method"].value_counts())

print("\nPredicted Intrusion Type Summary:")
print(results_df["Predicted_Intrusion_Type"].value_counts())

Traffic Status Summary:
Traffic_Status
INTRUSION    454626
BENIGN        89108
Name: count, dtype: int64

Detection Method Summary:
Detection_Method
Anomaly-Based      287731
Signature-Based    256003
Name: count, dtype: int64

Predicted Intrusion Type Summary:
Predicted_Intrusion_Type
Signature_Detected_Attack     256003
DDoS                           98154
BENIGN                         89108
DoS Hulk                       69849
DoS GoldenEye                  10200
DoS slowloris                   5107
DoS Slowhttptest                5032
FTP-Patator                     3974
SSH-Patator                     2961
Web Attack - Brute Force        1293
Bot                             1153
Web Attack - XSS                 622
PortScan                         265
Web Attack - Sql Injection        13
Name: count, dtype: int64


In [16]:
# 16) Save hybrid prediction results

output_dir = "../data/predictions"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "hybrid_intrusion_predictions.csv")
results_df.to_csv(output_path, index=False)

print(f"Hybrid predictions saved to: {output_path}")

Hybrid predictions saved to: ../data/predictions\hybrid_intrusion_predictions.csv
